# forced_alignment

> Routes for triggering forced alignment, polling progress, and toggling between NLTK and force-aligned pre-splits

In [ ]:
#| default_exp routes.forced_alignment

In [ ]:
#| export
from typing import Any, Dict, List, Tuple, Callable, Optional
from dataclasses import asdict

from fasthtml.common import Div, Span, Button, Progress, APIRouter

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_styles
from cjm_fasthtml_daisyui.components.feedback.loading import loading, loading_styles, loading_sizes
from cjm_fasthtml_daisyui.components.feedback.progress import progress as progress_cls, progress_colors
from cjm_fasthtml_daisyui.components.layout.join import join, join_item
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors, badge_sizes
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items, gap, justify
from cjm_fasthtml_tailwind.core.base import combine_classes
from cjm_fasthtml_lucide_icons.factory import lucide_icon

from cjm_workflow_state.state_store import SQLiteWorkflowStateStore
from cjm_transcript_segmentation.models import TextSegment, SegmentationUrls
from cjm_transcript_segmentation.components.step_renderer import render_seg_column_body
from cjm_transcript_segmentation.components.card_stack_config import SEG_CS_CONFIG, SEG_CS_IDS
from cjm_transcript_vad_align.models import VADChunk
from cjm_transcript_source_select.services.source import SourceService

from cjm_transcript_segment_align.components.step_renderer import (
    render_alignment_status, render_seg_mini_stats_badge,
)
from cjm_transcript_segment_align.services.forced_alignment import ForcedAlignmentService

## HTML IDs for Forced Alignment UI

In [ ]:
#| export
from cjm_transcript_segment_align.html_ids import CombinedHtmlIds

# Convenience aliases for FA HTML IDs
FA_CONTAINER_ID = CombinedHtmlIds.FA_CONTROLS
FA_STATUS_ID = CombinedHtmlIds.FA_STATUS

## UI Rendering Functions

In [ ]:
#| export
def render_fa_trigger_button(
    trigger_url: str,  # URL for forced alignment trigger route
    disabled: bool = False,  # Whether button is disabled
) -> Any:  # Force Align trigger button
    """Render the Force Align trigger button."""
    return Button(
        lucide_icon("audio-waveform", size=4, cls=str(m.r(2))),
        "Force Align",
        cls=combine_classes(btn, btn_colors.primary, btn_sizes.sm),
        disabled="disabled" if disabled else None,
        hx_post=trigger_url,
        hx_swap="none",
    )

In [ ]:
#| export
def render_fa_progress(
    progress_val: float,  # Progress value 0.0-1.0
    message: str,  # Progress stage message
    progress_url: str,  # URL for progress polling
) -> Any:  # Progress indicator with polling
    """Render forced alignment progress indicator with HTMX polling."""
    pct = int(progress_val * 100)
    return Div(
        Div(
            Span(cls=combine_classes(loading, loading_styles.spinner, loading_sizes.sm)),
            Span(
                f"{message} ({pct}%)",
                cls=combine_classes(font_size.sm, m.l(2))
            ),
            cls=combine_classes(flex_display, items.center)
        ),
        Progress(
            value=str(pct), max="100",
            cls=combine_classes(progress_cls, progress_colors.primary, "w-48")
        ),
        hx_get=progress_url,
        hx_trigger="every 1s",
        hx_target=f"#{FA_CONTAINER_ID}",
        hx_swap="innerHTML",
        cls=combine_classes(flex_display, items.center, gap(3)),
    )

In [ ]:
#| export
def render_fa_toggle(
    active_presplit: str,  # "nltk" or "forced_alignment"
    toggle_url: str,  # URL for toggle route
) -> Any:  # Toggle button group
    """Render the NLTK / Force Aligned toggle button group."""
    nltk_active = active_presplit == "nltk"
    fa_active = active_presplit == "forced_alignment"

    nltk_cls = combine_classes(
        btn, btn_sizes.sm, join_item,
        btn_colors.primary if nltk_active else btn_styles.outline
    )
    fa_cls = combine_classes(
        btn, btn_sizes.sm, join_item,
        btn_colors.primary if fa_active else btn_styles.outline
    )

    return Div(
        Span("Pre-splits:", cls=combine_classes(font_size.sm, m.r(2))),
        Div(
            Button(
                lucide_icon("text-cursor-input", size=4, cls=str(m.r(1))),
                "NLTK",
                cls=nltk_cls,
                hx_post=toggle_url,
                hx_vals='{"mode": "nltk"}',
                hx_swap="none",
                disabled="disabled" if nltk_active else None,
            ),
            Button(
                lucide_icon("audio-waveform", size=4, cls=str(m.r(1))),
                "Force Aligned",
                cls=fa_cls,
                hx_post=toggle_url,
                hx_vals='{"mode": "forced_alignment"}',
                hx_swap="none",
                disabled="disabled" if fa_active else None,
            ),
            cls=join,
        ),
        cls=combine_classes(flex_display, items.center),
    )

In [ ]:
#| export
def render_fa_controls(
    trigger_url: str = "",  # URL for trigger route
    toggle_url: str = "",  # URL for toggle route
    active_presplit: Optional[str] = None,  # Current active mode (None = no FA done yet)
    fa_available: bool = False,  # Whether FA plugin is available
    oob: bool = False,  # Whether to render as OOB swap
) -> Any:  # FA controls container
    """Render the forced alignment controls container.
    
    Shows either:
    - Trigger button (if FA not yet run)
    - Toggle (if FA has been run)
    - Nothing (if FA plugin not available)
    """
    if not fa_available:
        content = None
    elif active_presplit is not None:
        # FA has been run — show toggle
        content = render_fa_toggle(active_presplit, toggle_url)
    else:
        # FA available but not yet run — show trigger button
        content = render_fa_trigger_button(trigger_url)

    return Div(
        content,
        id=FA_CONTAINER_ID,
        cls=combine_classes(flex_display, items.center, gap(2)),
        hx_swap_oob="true" if oob else None,
    )

## Route Handlers

In [ ]:
#| export
async def _handle_fa_trigger(
    state_store: SQLiteWorkflowStateStore,
    workflow_id: str,
    fa_service: ForcedAlignmentService,
    source_service: SourceService,
    request: Any,
    sess: Any,
    seg_urls: SegmentationUrls,
    progress_url: str,
    toggle_url: str,
) -> Any:  # OOB updates for card stack, alignment status, FA controls, mini-stats
    """Trigger forced alignment and replace working segments."""
    session_id = get_session_id(sess)
    state = state_store.get_state(workflow_id, session_id)
    step_states = state.get("step_states", {})
    seg_state = step_states.get("segmentation", {})
    align_state = step_states.get("alignment", {})
    selection_state = step_states.get("selection", {})

    # Get VAD chunks (must be initialized)
    vad_chunk_dicts = align_state.get("vad_chunks", [])
    if not vad_chunk_dicts:
        return render_fa_controls(
            trigger_url="", fa_available=True, oob=True
        )

    vad_chunks = [VADChunk.from_dict(c) for c in vad_chunk_dicts]

    # Get source info for audio paths and text
    selected_sources = selection_state.get("selected_sources", [])
    source_blocks = source_service.get_source_blocks(selected_sources)

    # Build audio paths from source blocks
    audio_paths = []
    for src in selected_sources:
        block = source_service.get_transcription_by_id(src["record_id"], src["provider_id"])
        audio_paths.append(block.media_path)

    # Group VAD chunks by source (using audio_file_index if available)
    if len(source_blocks) == 1:
        vad_chunks_by_source = [vad_chunks]
    else:
        chunks_by_idx: Dict[int, List[VADChunk]] = {}
        for chunk in vad_chunks:
            idx = getattr(chunk, 'audio_file_index', 0)
            if idx not in chunks_by_idx:
                chunks_by_idx[idx] = []
            chunks_by_idx[idx].append(chunk)
        vad_chunks_by_source = [chunks_by_idx.get(i, []) for i in range(len(source_blocks))]

    # Run forced alignment
    fa_segments = await fa_service.align_and_split_combined_async(
        source_blocks=source_blocks,
        audio_paths=audio_paths,
        vad_chunks_by_source=vad_chunks_by_source,
    )

    # Save NLTK segments before replacing (preserve for toggle)
    if "nltk_segments" not in seg_state:
        seg_state["nltk_segments"] = seg_state.get("segments", [])
        seg_state["nltk_initial_segments"] = seg_state.get("initial_segments", [])

    # Store FA segments and set as active
    fa_seg_dicts = [asdict(s) for s in fa_segments]
    seg_state["fa_segments"] = fa_seg_dicts
    seg_state["fa_initial_segments"] = fa_seg_dicts[:]
    seg_state["segments"] = fa_seg_dicts
    seg_state["initial_segments"] = fa_seg_dicts[:]
    seg_state["active_presplit"] = "forced_alignment"
    seg_state["focused_index"] = 0
    seg_state["history"] = []

    step_states["segmentation"] = seg_state
    state["step_states"] = step_states
    state_store.update_state(workflow_id, session_id, state)

    # Re-render card stack and add OOB outerHTML for column body swap
    column_body = render_seg_column_body(
        segments=fa_segments,
        focused_index=0,
        visible_count=seg_state.get("visible_count", 3),
        card_width=seg_state.get("card_width", 80),
        urls=seg_urls,
        kb_system=None,
    )
    column_body.attrs['hx-swap-oob'] = 'outerHTML'

    # Build OOB updates
    segment_count = len(fa_segments)
    chunk_count = len(vad_chunk_dicts)

    alignment_status_oob = render_alignment_status(segment_count, chunk_count, oob=True)
    mini_stats_oob = render_seg_mini_stats_badge(fa_segments, oob=True)
    fa_controls_oob = render_fa_controls(
        toggle_url=toggle_url,
        active_presplit="forced_alignment",
        fa_available=True,
        oob=True,
    )

    return (column_body, alignment_status_oob, mini_stats_oob, fa_controls_oob)

In [ ]:
#| export
async def _handle_fa_toggle(
    state_store: SQLiteWorkflowStateStore,
    workflow_id: str,
    request: Any,
    sess: Any,
    seg_urls: SegmentationUrls,
    toggle_url: str,
) -> Any:  # OOB updates for card stack, alignment status, FA controls, mini-stats
    """Toggle between NLTK and force-aligned pre-splits."""
    form = await request.form()
    target_mode = form.get("mode", "nltk")

    session_id = get_session_id(sess)
    state = state_store.get_state(workflow_id, session_id)
    step_states = state.get("step_states", {})
    seg_state = step_states.get("segmentation", {})
    align_state = step_states.get("alignment", {})

    current_mode = seg_state.get("active_presplit", "nltk")

    if target_mode == current_mode:
        return  # No-op

    # Save current working segments to the current mode's storage
    if current_mode == "nltk":
        seg_state["nltk_segments"] = seg_state.get("segments", [])
    else:
        seg_state["fa_segments"] = seg_state.get("segments", [])

    # Restore target mode's segments
    if target_mode == "nltk":
        seg_state["segments"] = seg_state.get("nltk_segments", seg_state.get("initial_segments", []))
        seg_state["initial_segments"] = seg_state.get("nltk_initial_segments", seg_state.get("initial_segments", []))
    else:
        seg_state["segments"] = seg_state.get("fa_segments", [])
        seg_state["initial_segments"] = seg_state.get("fa_initial_segments", [])

    seg_state["active_presplit"] = target_mode
    seg_state["focused_index"] = 0
    seg_state["history"] = []

    step_states["segmentation"] = seg_state
    state["step_states"] = step_states
    state_store.update_state(workflow_id, session_id, state)

    # Re-render card stack with new segments and add OOB outerHTML
    segments = [TextSegment.from_dict(s) for s in seg_state["segments"]]

    column_body = render_seg_column_body(
        segments=segments,
        focused_index=0,
        visible_count=seg_state.get("visible_count", 3),
        card_width=seg_state.get("card_width", 80),
        urls=seg_urls,
        kb_system=None,
    )
    column_body.attrs['hx-swap-oob'] = 'outerHTML'

    # Build OOB updates
    chunk_count = len(align_state.get("vad_chunks", []))

    alignment_status_oob = render_alignment_status(len(segments), chunk_count, oob=True)
    mini_stats_oob = render_seg_mini_stats_badge(segments, oob=True)
    fa_controls_oob = render_fa_controls(
        toggle_url=toggle_url,
        active_presplit=target_mode,
        fa_available=True,
        oob=True,
    )

    return (column_body, alignment_status_oob, mini_stats_oob, fa_controls_oob)

## Router Assembly

In [ ]:
#| export
def init_forced_alignment_routers(
    state_store: SQLiteWorkflowStateStore,  # State store instance
    workflow_id: str,  # Workflow identifier
    fa_service: ForcedAlignmentService,  # Forced alignment service
    source_service: SourceService,  # Source service for audio paths/text
    seg_urls: SegmentationUrls,  # Segmentation URL bundle
    prefix: str,  # Route prefix (e.g., "/fa")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize forced alignment routes."""
    router = APIRouter(prefix=prefix)

    @router
    async def trigger(request, sess):
        """Trigger forced alignment."""
        return await _handle_fa_trigger(
            state_store, workflow_id, fa_service, source_service,
            request, sess, seg_urls,
            progress_url=progress.to(),
            toggle_url=toggle.to(),
        )

    @router
    async def progress(request, sess):
        """Poll forced alignment progress."""
        # Check if FA is still running by checking plugin progress
        proxy = fa_service._manager.get_plugin(fa_service._plugin_name)
        if proxy:
            try:
                prog = await proxy.get_progress_async()
                if prog and prog.get("progress", 0) < 1.0:
                    return render_fa_progress(
                        progress_val=prog["progress"],
                        message=prog.get("message", "Processing..."),
                        progress_url=progress.to(),
                    )
            except Exception:
                pass

        # FA is done or not running — show trigger button (will be replaced by toggle after trigger completes)
        return render_fa_trigger_button(trigger.to())

    @router
    async def toggle(request, sess):
        """Toggle between NLTK and force-aligned pre-splits."""
        return await _handle_fa_toggle(
            state_store, workflow_id,
            request, sess, seg_urls,
            toggle_url=toggle.to(),
        )

    routes = {
        "trigger": trigger,
        "progress": progress,
        "toggle": toggle,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()